In [ ]:
from google.colab import drive
import sys
import os

# 1. Montar Drive
drive.mount('/content/drive')

# 2. CAMBIAR DIRECTORIO (Esto es lo clave)
# Ajusta esta ruta a donde tengas tu carpeta del proyecto
project_path = '/content/drive/MyDrive/Proyecto Final MIR'
%cd {project_path}

# 3. Agregar el proyecto al path de Python para que encuentre 'src'
sys.path.append(project_path)

# 4. Verificar
print("Estás parado en:", os.getcwd())
# Deberías ver tus carpetas 'data', 'src', etc.
!ls

Mounted at /content/drive
/content/drive/MyDrive/Proyecto Final MIR
Estás parado en: /content/drive/MyDrive/Proyecto Final MIR
data  EDA.ipynb  embeddings  environment.yaml  reports	Reports  src


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

# ================= IMPORTACIONES =================
try:
    from src.Models.definitions import AudioNetwork
    from src.Models.utils import get_dataloaders
except ImportError:
    from Models.definitions import AudioNetwork
    from Models.utils import get_dataloaders

# ================= CONFIGURACIÓN =================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 30

WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 5

BASE_OUTPUT = Path("reports/audio_expert")
MODEL_SAVE_PATH = Path("src/saved_models")
MODEL_NAME = "audio_network_best.pth"

CLASS_NAMES = ["Happy (Q1)", "Angry (Q2)", "Sad (Q3)", "Relaxed (Q4)"]
# ============================================================


# ============================================================
#               TRAIN ONE EPOCH (CON GRADIENT CLIPPING)
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, clip_value=1.0):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    loop = tqdm(loader, desc="Entrenando", leave=False)

    for batch in loop:
        x_2d = batch['audio_2d'].to(DEVICE)
        x_1d = batch['audio_1d'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(x_2d, x_1d)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(loss=loss.item())

    return running_loss / len(loader), 100 * correct / total


# ============================================================
#                        VALIDACIÓN
# ============================================================
def validate(model, loader, criterion, return_preds=False):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0

    all_preds, all_labels, all_probs, all_ids = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_2d = batch['audio_2d'].to(DEVICE)
            x_1d = batch['audio_1d'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            ids = batch.get("spotify_id", [])

            outputs = model(x_2d, x_1d)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            if return_preds:
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_ids.extend(ids)

    if return_preds:
        return running_loss / len(loader), 100 * correct / total, all_labels, all_preds, all_probs, all_ids

    return running_loss / len(loader), 100 * correct / total


# ============================================================
#                     PLOTS & CONFUSION MATRIX
# ============================================================
def save_plots(history, output_dir):
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title("Curvas de Loss (Audio)")
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title("Curvas de Accuracy (Audio)")
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(output_dir / "metrics_plot_audio.png")
    plt.close()


def save_confusion_matrix(y_true, y_pred, output_dir):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicción")
    plt.ylabel("Realidad")
    plt.title("Matriz de Confusión — Audio")
    plt.savefig(output_dir / "confusion_matrix_audio.png")
    plt.close()


# ============================================================
#                           MAIN
# ============================================================
def main():
    BASE_OUTPUT.mkdir(parents=True, exist_ok=True)
    MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

    print(f"\n🎧 Entrenando Audio Expert en: {DEVICE}")

    # --------------------------
    # 1. Cargar DataLoaders
    # --------------------------
    train_loader, val_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
    if train_loader is None:
        return

    # --------------------------
    # 2. CLASS WEIGHTS (rápido)
    # --------------------------
    print("⚖️ Calculando class weights desde master_dataset.csv...")

    master_csv = Path("data/processed/master_dataset.csv")
    df = pd.read_csv(master_csv)

    df_train = df[df["split"] == "train"]

    label_map = {
        "Q1_Happy": 0,
        "Q2_Angry": 1,
        "Q3_Sad": 2,
        "Q4_Relaxed": 3,
    }

    numeric_labels = df_train["label_quadrant"].map(label_map).astype(int)
    class_counts = numeric_labels.value_counts().sort_index()
    class_counts = torch.tensor(class_counts.values, dtype=torch.float32)

    print("   → Frecuencias:", class_counts.tolist())

    class_weights = 1.0 / class_counts
    class_weights = class_weights * (4 / class_weights.sum())
    class_weights = class_weights.to(DEVICE)

    print("   → Class weights:", class_weights.tolist())

    # --------------------------
    # 3. Modelo + Optimización
    # --------------------------
    model = AudioNetwork(num_classes=4, audio_1d_dim=34).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2)

    history = {"epoch": [], "train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": [], "lr": []}

    best_val_acc = 0
    epochs_no_improve = 0

    # --------------------------
    # 4. TRAINING LOOP
    # --------------------------
    print("\n🚀 Comenzando entrenamiento...\n")

    for epoch in range(EPOCHS):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        v_loss, v_acc = validate(model, val_loader, criterion)

        current_lr = optimizer.param_groups[0]['lr']

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(t_loss)
        history["train_acc"].append(t_acc)
        history["val_loss"].append(v_loss)
        history["val_acc"].append(v_acc)
        history["lr"].append(current_lr)

        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"T.Loss={t_loss:.4f} Acc={t_acc:.1f}% | "
              f"V.Loss={v_loss:.4f} Acc={v_acc:.1f}% | LR={current_lr:.5f}")

        # Guardar mejor modelo
        if v_acc > best_val_acc + 1e-4:
            best_val_acc = v_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH / MODEL_NAME)
        else:
            epochs_no_improve += 1

        scheduler.step(v_acc)

        # Early stopping
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print("\n⛔ Early Stopping activado.")
            break

    # --------------------------
    # 5. REPORTES
    # --------------------------
    print("\n📊 Entrenamiento finalizado.")
    print(f"🏆 Mejor Val Accuracy: {best_val_acc:.2f}%")

    pd.DataFrame(history).to_csv(BASE_OUTPUT / "training_log_audio.csv", index=False)
    save_plots(history, BASE_OUTPUT)

    # Cargar mejor modelo
    model.load_state_dict(torch.load(MODEL_SAVE_PATH / MODEL_NAME))

    # --------------------------
    # Exportar predicciones
    # --------------------------
    def exportar(loader, filename, name):
        print(f"→ Exportando {name}...")
        _, acc, y_true, y_pred, y_probs, y_ids = validate(
            model, loader, criterion, return_preds=True
        )

        df = pd.DataFrame(
            y_probs,
            columns=["prob_audio_Q1", "prob_audio_Q2",
                     "prob_audio_Q3", "prob_audio_Q4"]
        )
        df.insert(0, "spotify_id", y_ids)
        df["true_label"] = y_true

        df.to_csv(BASE_OUTPUT / filename, index=False)
        return y_true, y_pred

    exportar(train_loader, "predicciones_audio_TRAIN.csv", "TRAIN")
    exportar(val_loader, "predicciones_audio_VAL.csv", "VAL")
    y_true_test, y_pred_test = exportar(test_loader, "predicciones_audio_TEST.csv", "TEST")

    save_confusion_matrix(y_true_test, y_pred_test, BASE_OUTPUT)

    with open(BASE_OUTPUT / "final_report_audio.txt", "w") as f:
        f.write(classification_report(y_true_test, y_pred_test, target_names=CLASS_NAMES))

    print("\n📂 ¡Proceso completado!")
    print(classification_report(y_true_test, y_pred_test, target_names=CLASS_NAMES))


if __name__ == "__main__":
    main()



🎧 Entrenando Audio Expert en: cuda
🚀 Creando DataLoaders (Batch: 32) | Chi2: True
🔍 Ejecutando selección de características (Chi^2) sobre TRAIN...
✅ Selección completada: 2000 -> 500 features más relevantes.
   -> Usando 500 features de texto seleccionadas (Chi^2).
   ✅ Train: 4518 | Val: 968 | Test: 969
⚖️ Calculando class weights desde master_dataset.csv...
   → Frecuencias: [1390.0, 1393.0, 1391.0, 344.0]
   → Class weights: [0.5683574080467224, 0.567133367061615, 0.5679488182067871, 2.296560525894165]

🚀 Comenzando entrenamiento...



Epoch 1/30 | T.Loss=1.3995 Acc=52.0% | V.Loss=1.1109 Acc=40.6% | LR=0.00100


Epoch 2/30 | T.Loss=0.8648 Acc=66.8% | V.Loss=0.9340 Acc=59.1% | LR=0.00100


Epoch 3/30 | T.Loss=0.7854 Acc=70.1% | V.Loss=0.9157 Acc=66.7% | LR=0.00100


Epoch 4/30 | T.Loss=0.7672 Acc=70.4% | V.Loss=0.7993 Acc=67.0% | LR=0.00100


...

Epoch 9/30 | T.Loss=0.6735 Acc=74.8% | V.Loss=0.8040 Acc=67.8% | LR=0.00100


Epoch 10/30 | T.Loss=0.6670 Acc=74.3% | V.Loss=0.9707 Acc=60.8% | LR=0.00100


Epoch 11/30 | T.Loss=0.6484 Acc=75.4% | V.Loss=0.7064 Acc=71.6% | LR=0.00100


Epoch 12/30 | T.Loss=0.5714 Acc=78.2% | V.Loss=0.7873 Acc=71.1% | LR=0.00050


Epoch 13/30 | T.Loss=0.5572 Acc=78.9% | V.Loss=0.7849 Acc=69.3% | LR=0.00050

⛔ Early Stopping activado.

📊 Entrenamiento finalizado.
🏆 Mejor Val Accuracy: 72.62%
→ Exportando TRAIN...
→ Exportando VAL...
→ Exportando TEST...

📂 ¡Proceso completado!
              precision    recall  f1-score   support

  Happy (Q1)       0.77      0.61      0.68       298
  Angry (Q2)       0.70      0.69      0.69       299
    Sad (Q3)       0.83      0.82      0.82       298
Relaxed (Q4)       0.40      0.77      0.53        74

    accuracy                           0.71       969
   macro avg       0.67      0.72      0.68       969
weighted avg       0.74      0.71      0.72       969

